In [7]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

sys.path.append(os.path.dirname(os.getcwd()))
from lib.utils import get_sequence_data
from lib.BLogistic import SkewedBLogistic

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

folder_path = r"../../MarketData/historical_data"
context_window = 60
X, Y     = get_sequence_data(folder_path, context_window, force_recompute=True)
dof      = 16
simple_X = torch.tensor(X[:, :, 0], device=device)
simple_Y = torch.tensor(Y, device=device).reshape(-1, 1)

dev_size = 10000
np.random.seed(0)
indices = np.random.permutation(simple_X.shape[0])

dev_indices = indices[:dev_size]
train_indices = indices[dev_size:]
train_X = simple_X[train_indices]
train_Y = simple_Y[train_indices]
dev_X = simple_X[dev_indices, :]
dev_Y = simple_Y[dev_indices]

std = train_Y.std()
train_X = train_X / std
train_Y = train_Y / std
dev_X = dev_X / std
dev_Y = dev_Y / std
print("train_X", train_X.shape, "train_Y", train_Y.shape, "dev_X", dev_X.shape, "dev_Y", dev_Y.shape)
print("std", std, train_X.std())

skipping 2020-11-27
skipping 2020-12-24
skipping 2021-11-26
skipping 2022-11-25
skipping 2023-07-03
skipping 2023-11-24
skipping 2024-07-03
skipping 2024-11-29
skipping 2024-12-24
skipping 2025-07-03
train_X torch.Size([402426, 60]) train_Y torch.Size([402426, 1]) dev_X torch.Size([10000, 60]) dev_Y torch.Size([10000, 1])
std tensor(0.0004, device='cuda:0') tensor(1.0232, device='cuda:0')


In [11]:
class LSTMProbNN(nn.Module):
    def __init__(self, context_window, dof, device):
        super().__init__()
        self.context_window = context_window
        self.dof            = dof
        self.device         = device

        print(f"\nInitializing LSTM with context_window={context_window}, dof={dof}")

        # LSTM layers (3 layers with decreasing neurons: 128 -> 64 -> 32)
        self.lstm1 = nn.LSTM(
            input_size=1,
            hidden_size=128,
            num_layers=1,
            batch_first=True,
            dropout=0
        )

        self.lstm2 = nn.LSTM(
            input_size=128,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            dropout=0
        )

        self.lstm3 = nn.LSTM(
            input_size=64,
            hidden_size=32,
            num_layers=1,
            batch_first=True,
            dropout=0
        )

        self.dropout = nn.Dropout(0.02)

        self.fc = nn.Linear(32, dof)
        nn.init.uniform_(self.fc.weight, -0.01, 0.01)
        nn.init.zeros_(self.fc.bias)

        self.blogistic = SkewedBLogistic(dof - 3, device=device)


    def forward(self, x):

        if x.dim() == 2:
            x = x.unsqueeze(-1)
        elif x.dim() == 3 and x.shape[1] == 1:
            x = x.transpose(1, 2)

        x, _ = self.lstm1(x)
        x = self.dropout(x)

        x, _ = self.lstm2(x)
        x = self.dropout(x)

        x, _ = self.lstm3(x)
        x = self.dropout(x)

        x = x[:, -1, :]

        params = self.fc(x)
        return params

    def get_params(self, x):
        return self.forward(x)

    def get_logpdf(self, x, sample_xs):
        params = self.get_params(x)
        return self.blogistic.logpdf(sample_xs, params[:, :-2].flatten(),
                                     params[:, -2], params[:, -1])

    def loss_fn(self, x, y):
        params = self.forward(x)
        logpdf = self.blogistic.logpdf_vectorized(
            y, params[:, :-2], params[:, -2], params[:, -1]
        )
        return -logpdf.mean()

    def get_pdf(self, x, sample_xs):
        return torch.exp(self.get_logpdf(x, sample_xs))

In [12]:
def train_lstm_nn(
    train_X, train_Y, dev_X, dev_Y,
    context_window, dof, lr, num_epochs, device,
    batch_size=128
):
    print(f"Train X shape: {train_X.shape}, Train Y shape: {train_Y.shape}")
    print(f"Using batch_size={batch_size}, lr={lr}, context_window={context_window}, dof={dof}")

    model = LSTMProbNN(context_window, dof, device).to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.002)
    train_losses, dev_losses = [], []

    # Training loop
    for epoch in range(num_epochs):

        model.train()
        running_loss = 0.0
        n_batches = 0


        for i in range(0, train_X.shape[0], batch_size):
            batch_X = train_X[i:i+batch_size]
            batch_Y = train_Y[i:i+batch_size]

            if batch_X.shape[0] == 0:
                continue

            loss = model.loss_fn(batch_X, batch_Y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            n_batches += 1

        avg_train_loss = running_loss / max(n_batches, 1)

        if epoch % 100 == 0:
            model.eval()
            with torch.no_grad():
                dev_loss_sum = 0.0
                n_dev_batches = 0
                for j in range(0, dev_X.shape[0], batch_size):
                    batch_X = dev_X[j:j+batch_size]
                    batch_Y = dev_Y[j:j+batch_size]
                    if batch_X.shape[0] == 0:
                        continue
                    dev_loss_sum += model.loss_fn(batch_X, batch_Y).item()
                    n_dev_batches += 1
                avg_dev_loss = dev_loss_sum / max(n_dev_batches, 1)


            train_losses.append(avg_train_loss)
            dev_losses.append(avg_dev_loss)
            print(f"[Step {epoch}] Train Loss: {avg_train_loss:.4f}, Dev Loss: {avg_dev_loss:.4f}")

    return model, train_losses, dev_losses

In [ ]:
print(f"Context window: {context_window}")
print(f"DOF: {dof}")
print(f"Device: {device}")

lr = 2e-4
num_steps = 300

model, train_losses, dev_losses = train_lstm_nn(train_X, train_Y, dev_X, dev_Y,
                                                context_window, dof,
                                                lr, num_steps, device=device)

Context window: 60
DOF: 16
Device: cuda
Train X shape: torch.Size([402426, 60]), Train Y shape: torch.Size([402426, 1])
Using batch_size=128, lr=0.0002, context_window=60, dof=16

Initializing LSTM with context_window=60, dof=16
[Step 0] Train Loss: 1.1509, Dev Loss: 1.1208


In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

plot_xs = torch.linspace(-8, 8, 10000, device=device)
nplots = 10

for idx in range(nplots):
    # get color wheel
    color = plt.cm.viridis(idx / nplots)
    plot_ys = model.get_pdf(dev_X[idx, :].reshape(1, -1), plot_xs)
    plt.plot(plot_xs.cpu().numpy(), plot_ys.detach().cpu().numpy(), color=color)
    plt.scatter(dev_Y[idx, :].cpu().numpy(), model.get_pdf(dev_X[idx, :].reshape(1, -1), dev_Y[idx, :].reshape(1, 1)).detach().cpu().numpy(), color=color)

plt.xlabel("Return")
plt.ylabel("PDF")
plt.show()

Context window: 60
DOF: 16
Device: cuda
Train X: torch.Size([402426, 60]), dtype: torch.float32
Train Y: torch.Size([402426, 1]), dtype: torch.float32
LSTM initialized: [128, 64, 32], dropout=0.02, L2=0.002


C:\Users\MainUser\miniconda3\envs\cs231_env\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.02 and num_layers=1
  warnings.warn(


RuntimeError: expected scalar type Double but found Float